# Goalkeeper Analysis System Documentation

## Overview
The Goalkeeper Analysis System is a sophisticated analytics tool designed to evaluate goalkeeper performance in football/soccer using StatsBomb data. The system provides comprehensive analysis across four key performance metrics and generates normalized scores for comparison across different leagues and seasons.

## Key Features
- Multi-league and multi-season analysis capability
- Four distinct performance metrics evaluation
- Normalized scoring system (4-10 scale)
- Confidence interval calculations
- Minutes-played weighted calculations
- Comprehensive logging system

## Core Performance Metrics

### 1. Shot Stopping (calculate_shot_stopping)
Evaluates a goalkeeper's ability to prevent goals.

**Components:**
- Save percentage calculation
- Expected Goals (xG) analysis
- Tournament-normalized Post-Shot Expected Goals (PSxG)
- Score Range: 0-1 (normalized to 4-10 in final output)

**Formula:**
```
Final Score = (save_percentage * 0.4) + (normalized_psxg * 0.6)
```

### 2. Distribution (calculate_distribution)
Assesses a goalkeeper's passing and ball distribution abilities.

**Components:**
- Short pass completion rate (<30 yards)
- Long pass completion rate (≥30 yards)
- Progressive passes analysis
- Pressure consideration factor

**Formula:**
```
Final Score = (short_completion + long_completion + prog_completion) / 3 * pressure_factor
```

### 3. Sweeping (calculate_sweeping)
Evaluates a goalkeeper's ability to act as a sweeper-keeper.

**Components:**
- Sweeping actions identification
- Danger level calculation with exponential distance decay
- Success value assessment
- Coverage score calculation
- Volume bonus factor

**Formula:**
```
Final Score = (weighted_score * 0.7 + coverage_score * 0.3) * (0.7 + 0.3 * volume_factor)
```

### 4. Aerial Command (calculate_aerial_command)
Measures a goalkeeper's ability to handle aerial threats.

**Components:**
- Cross claim success rate
- Punch success rate
- Intervention rate
- Opposition cross completion rate

**Formula:**
```
Final Score = (intervention_rate * 0.4) + (action_success * 0.4) + ((1 - opposition_success_rate) * 0.2)
```

## Data Processing Pipeline

### 1. Data Collection
- Utilizes StatsBomb's API through `statsbombpy`
- Collects event-level data for each match
- Filters for goalkeeper-specific events

### 2. Performance Calculation
1. Individual match analysis
2. Metric calculation for each performance aspect
3. Normalization of raw scores
4. Confidence interval calculation
5. Composite score generation

### 3. Data Aggregation
- Groups data by goalkeeper
- Calculates weighted averages based on minutes played
- Generates league and season-specific analyses
- Consolidates multi-league data

## Score Normalization System

### Normalization Process
1. **Raw Score Collection**
   - Collects raw metric scores (0-1 scale)
   - Considers minutes played for reliability

2. **Statistical Adjustments**
   - Applies reliability factors based on minutes played
   - Uses percentile-based bounds for larger samples
   - Implements min/max padding for smaller samples

3. **Final Score Calculation**
   - Converts to 4-10 scale
   - Adjusts range based on sample reliability
   - Includes confidence interval calculations

## Usage Instructions

### Basic Usage
```python
# Single league analysis
results = analyze_goalkeeper_performance(competition_id, season_id)

# Multi-league analysis
leagues = [
    {"competition_id": 9, "season_ids": [281]},
    {"competition_id": 43, "season_ids": [106]}
]
results = analyze_multiple_leagues(leagues)
```

### Output Format
The system generates a DataFrame with the following columns:
- goalkeeper: Player name
- team: Team name
- competition_name: League name
- season_id: Season identifier
- matches_played: Number of matches
- minutes_played: Total minutes
- shot_stopping: Normalized score (4-10)
- distribution: Normalized score (4-10)
- sweeping: Normalized score (4-10)
- aerial_command: Normalized score (4-10)
- composite_score: Overall performance score

## Error Handling and Logging
- Comprehensive error logging system
- Match-level error handling
- Season-level error handling
- Progress tracking with tqdm
- Detailed logging to both file and console

## Technical Requirements
- Python 3.x
- Required packages:
  - statsbombpy
  - pandas
  - numpy
  - tqdm
  - logging

## System Limitations
1. Requires minimum 90 minutes played for reliable analysis
2. Dependent on StatsBomb data availability
3. Score normalization may vary with sample size
4. League-specific context not considered in normalization

## Best Practices
1. Use multiple seasons for more reliable analysis
2. Consider confidence intervals for small sample sizes
3. Account for league-specific contexts in interpretation
4. Review raw metrics alongside normalized scores

In [11]:
from statsbombpy import sb
import pandas as pd
import numpy as np
from tqdm import tqdm
import logging
import warnings
import glob
import os
warnings.simplefilter("ignore")


In [12]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('goalkeeper_analysis.log'),
        logging.StreamHandler()
    ]
)


In [13]:
pd.set_option('display.max_rows', None)

In [14]:
def calculate_shot_stopping(events, goalkeeper, team):
    """
    Calculate shot stopping performance metrics with improved normalization
    Returns a score between 0-1
    """
    shots_against = events[
        (events['type'] == 'Shot') & 
        (events['team'] != team)
    ]
    
    if len(shots_against) == 0:
        return 0
        
    saves = len(shots_against[shots_against['shot_outcome'] == 'Saved'])
    goals = len(shots_against[shots_against['shot_outcome'] == 'Goal'])
    xg = shots_against['shot_statsbomb_xg'].sum()
    
    # Calculate components with safety checks
    total_shots = saves + goals
    save_percentage = saves / total_shots if total_shots > 0 else 0
    
    # Calculate tournament mean xG for normalization
    all_shots = events[events['type'] == 'Shot']
    tournament_mean_xg = all_shots['shot_statsbomb_xg'].mean()
    
    # Normalize PSxG using tournament mean
    psxg_diff = xg - goals
    normalized_psxg = psxg_diff / (tournament_mean_xg * total_shots) if total_shots > 0 else 0
    normalized_psxg = max(-1, min(1, normalized_psxg))  # Clamp to [-1, 1]
    
    # Convert to 0-1 range
    normalized_psxg = (normalized_psxg + 1) / 2
    
    return (save_percentage * 0.4) + (normalized_psxg * 0.6)

In [15]:
def calculate_distribution(events, goalkeeper, team):
    """
    Calculate distribution performance metrics with progressive passes
    Returns a score between 0-1
    """
    gk_events = events[events['player'] == goalkeeper]
    gk_passes = gk_events[gk_events['type'] == 'Pass']
    
    if len(gk_passes) == 0:
        return 0
        
    # Separate short and long passes
    short_passes = gk_passes[gk_passes['pass_length'] < 30]
    long_passes = gk_passes[gk_passes['pass_length'] >= 30]
    
    # Calculate completion rates
    short_completion = (
        len(short_passes[short_passes['pass_outcome'].isna()]) / 
        len(short_passes) if len(short_passes) > 0 else 0
    )
    
    long_completion = (
        len(long_passes[long_passes['pass_outcome'].isna()]) / 
        len(long_passes) if len(long_passes) > 0 else 0
    )
    
    # Calculate progressive passes
    def is_progressive(pass_event):
        if not isinstance(pass_event['location'], list) or not isinstance(pass_event['pass_end_location'], list):
            return False
        
        start_y = pass_event['location'][1]
        end_y = pass_event['pass_end_location'][1]
        
        # Consider a pass progressive if it advances the ball at least 10 yards
        return (end_y - start_y) >= 10
    
    progressive_passes = gk_passes[gk_passes.apply(is_progressive, axis=1)]
    prog_completion = (
        len(progressive_passes[progressive_passes['pass_outcome'].isna()]) /
        len(progressive_passes) if len(progressive_passes) > 0 else 0
    )
    
    # Equal weighting between short, long, and progressive passes
    weighted_score = (short_completion + long_completion + prog_completion) / 3
    
    # Consider pressure
    pressure_events = len(events[
        (events['team'] != team) & 
        (events['type'] == 'Pressure')
    ])
    
    pressure_factor = min(1.2, 1 + (pressure_events / 100))
    
    return weighted_score * pressure_factor


In [16]:
def calculate_sweeping(events, goalkeeper, team):
    """
    Calculate sweeping performance metrics with improved calculations
    Returns a score between 0-1
    """
    gk_events = events[events['player'] == goalkeeper]
    
    # Define sweeping actions
    sweeping_actions = gk_events[
        ((gk_events['type'] == 'Goal Keeper') & 
         (gk_events['goalkeeper_type'] == 'Keeper Sweeper')) |
        ((gk_events['type'].isin(['Interception', 'Block'])) &
         (gk_events['location'].apply(lambda x: isinstance(x, list) and x[1] > 40)))
    ]
    
    # Calculate danger level with exponential distance decay
    def calculate_danger_level(event):
        if not isinstance(event['location'], list):
            return 0.5
        
        # Exponential decay for distance from goal
        distance_from_goal = abs(80 - event['location'][1])
        distance_factor = np.exp(-distance_from_goal / 20)  # Decay factor of 20
        
        # Consider horizontal position with normal distribution
        width_factor = np.exp(-((event['location'][0] - 40) ** 2) / 800)  # SD of 20
        
        return (distance_factor * 0.7 + width_factor * 0.3)

    # Calculate success with empirically-based values
    def calculate_success_value(action):
        outcomes = {
            'Success': 1.0,
            'Success In Play': 0.9,
            'Success Out': 0.8,
            'Clear': 0.7,
            'Fail': 0.0
        }
        
        if action['type'] == 'Goal Keeper':
            return outcomes.get(action['goalkeeper_outcome'], 0.5)
        return 0.9  # High value for preventive actions
    
    # Calculate potential sweeping opportunities
    through_balls = events[
        (events['team'] != team) & 
        (events['pass_type'].isin(['Through Ball', 'Pass'])) &
        (events['location'].apply(lambda x: isinstance(x, list) and x[1] > 40))
    ]
    
    def calculate_coverage_score():
        if len(through_balls) == 0:
            return 0.5
        
        intercepted = len(sweeping_actions)
        total_opportunities = len(through_balls)
        
        coverage_ratio = intercepted / total_opportunities
        return min(coverage_ratio * 1.5, 1.0)
    
    if len(sweeping_actions) == 0:
        return calculate_coverage_score() * 0.3
    
    # Calculate weighted scores
    danger_levels = [calculate_danger_level(action) for _, action in sweeping_actions.iterrows()]
    success_values = [calculate_success_value(action) for _, action in sweeping_actions.iterrows()]
    
    weighted_score = sum(d * s for d, s in zip(danger_levels, success_values)) / len(sweeping_actions)
    coverage_score = calculate_coverage_score()
    
    # Volume bonus based on logarithmic scale
    volume_factor = min(np.log1p(len(sweeping_actions)) / np.log1p(10), 1.0)
    
    return (weighted_score * 0.7 + coverage_score * 0.3) * (0.7 + 0.3 * volume_factor)

In [17]:
def calculate_aerial_command(events, goalkeeper, team):
    """
    Calculate aerial command metrics with improved calculations
    Returns a score between 0-1
    """
    gk_events = events[events['player'] == goalkeeper]
    
    # Consider all aerial threats
    crosses = events[
        (events['pass_cross'] == True) & 
        (events['team'] != team)
    ]
    
    if len(crosses) == 0:
        return 0
    
    # Calculate different types of aerial actions
    claims = len(gk_events[
        (gk_events['type'] == 'Goal Keeper') & 
        (gk_events['goalkeeper_type'] == 'Claim')
    ])
    
    punches = len(gk_events[
        (gk_events['type'] == 'Goal Keeper') & 
        (gk_events['goalkeeper_type'] == 'Punch')
    ])
    
    # Calculate success rates for different action types
    claim_success = len(gk_events[
        (gk_events['type'] == 'Goal Keeper') & 
        (gk_events['goalkeeper_type'] == 'Claim') &
        (gk_events['goalkeeper_outcome'].isin(['Success', 'Success In Play']))
    ]) / max(claims, 1)
    
    punch_success = len(gk_events[
        (gk_events['type'] == 'Goal Keeper') & 
        (gk_events['goalkeeper_type'] == 'Punch') &
        (gk_events['goalkeeper_outcome'].isin(['Success', 'Success In Play']))
    ]) / max(punches, 1)
    
    # Calculate intervention rate
    aerial_actions = claims + punches
    intervention_rate = aerial_actions / len(crosses)
    
    # Calculate opposition cross completion rate
    successful_crosses = len(crosses[crosses['pass_outcome'].isna()])
    opposition_success_rate = successful_crosses / len(crosses)
    
    # Combine metrics with empirical weights
    action_success = (claim_success * 0.6 + punch_success * 0.4) if aerial_actions > 0 else 0
    return (
        (intervention_rate * 0.4) + 
        (action_success * 0.4) + 
        ((1 - opposition_success_rate) * 0.2)
    )

In [18]:
def normalize_metric(series, min_val=4, max_val=10, minutes_played=None):
    """
    Improved metric normalization with better handling of edge cases
    """
    if len(series) == 0:
        return pd.Series([])
    
    if minutes_played is not None:
        # Increase minimum minutes threshold
        min_minutes = 90  # One full match minimum
        reliability_factor = np.minimum(minutes_played / (4 * 90), 1)  # Scale up to 4 matches
        weights = reliability_factor
        series = series * weights
    
    if series.std() < 1e-10:
        return pd.Series([6.5] * len(series))
    
    # Use more robust bounds based on sample size
    if len(series) < 10:
        # For small samples, use min/max with some padding
        lower_bound = series.min() - (series.std() * 0.5)
        upper_bound = series.max() + (series.std() * 0.5)
    else:
        # For larger samples, use percentiles
        lower_bound = series.quantile(0.05)  # 5th percentile
        upper_bound = series.quantile(0.95)  # 95th percentile
    
    # Normalize and scale
    normalized = (series - lower_bound) / (upper_bound - lower_bound)
    normalized = np.clip(normalized, 0, 1)
    
    # Dynamic range based on sample reliability
    if minutes_played is not None:
        # Compress range for less reliable samples
        min_val_adj = min_val + (1 - reliability_factor.mean()) * 2
        max_val_adj = max_val - (1 - reliability_factor.mean()) * 2
    else:
        min_val_adj = min_val
        max_val_adj = max_val
    
    scaled = normalized * (max_val_adj - min_val_adj) + min_val_adj
    
    # Add confidence intervals (as additional columns if needed)
    std_err = series.std() / np.sqrt(len(series))
    conf_interval = 1.96 * std_err  # 95% confidence interval
    
    return scaled.round(1), conf_interval

In [19]:
def analyze_goalkeeper_performance(competition_id, season_id):
    """
    Analyze goalkeeper performance across multiple metrics, aggregated across all matches
    Returns a DataFrame with normalized scores and confidence intervals
    
    Parameters:
    competition_id (int): ID of the competition to analyze
    season_id (int): ID of the season to analyze
    
    Returns:
    pandas.DataFrame: Aggregated goalkeeper statistics with normalized scores and confidence intervals
    """
    matches = sb.matches(competition_id=competition_id, season_id=season_id)
    all_gk_stats = []
    
    for _, match in matches.iterrows():
        try:
            events = sb.events(match_id=match['match_id'])
            goalkeepers = events[
                (events['position'] == 'Goalkeeper')
            ]['player'].unique()
            
            for gk in goalkeepers:
                gk_events = events[events['player'] == gk]
                if len(gk_events) == 0:
                    continue
                
                team = gk_events['team'].iloc[0]
                
                # Calculate minutes played including added time
                start_minute = gk_events['minute'].min()
                end_minute = gk_events['minute'].max()
                added_time = gk_events['second'].max() / 60 if end_minute >= 90 else 0
                minutes = end_minute - start_minute + added_time
                
                # Calculate all metrics
                metrics = {
                    'goalkeeper': gk,
                    'team': team,
                    'match_id': match['match_id'],
                    'minutes_played': minutes,
                    'shot_stopping': calculate_shot_stopping(events, gk, team),
                    'distribution': calculate_distribution(events, gk, team),
                    'sweeping': calculate_sweeping(events, gk, team),
                    'aerial_command': calculate_aerial_command(events, gk, team)
                }
                
                all_gk_stats.append(metrics)
                
        except Exception as e:
            print(f"Error processing match {match['match_id']}: {str(e)}")
            continue
    
    # Create DataFrame
    df = pd.DataFrame(all_gk_stats)
    if len(df) == 0:
        return pd.DataFrame()
    
    # Group by goalkeeper and calculate weighted averages
    metrics = ['shot_stopping', 'distribution', 'sweeping', 'aerial_command']
    
    def weighted_avg(x):
        return np.average(x, weights=df.loc[x.index, 'minutes_played'])
    
    grouped_df = df.groupby(['goalkeeper', 'team']).agg({
        'minutes_played': 'sum',
        **{metric: weighted_avg for metric in metrics}
    }).reset_index()
    
    # Add matches played and full matches equivalent
    matches_played = df.groupby('goalkeeper')['match_id'].nunique()
    grouped_df['matches_played'] = grouped_df['goalkeeper'].map(matches_played)
    grouped_df['full_match_equivalent'] = (grouped_df['minutes_played'] / 90).round(1)
    
    # Normalize all metrics and add confidence intervals
    normalized_metrics = {}
    confidence_intervals = {}
    
    for metric in metrics:
        normalized, conf_interval = normalize_metric(
            grouped_df[metric], 
            minutes_played=grouped_df['minutes_played']
        )
        normalized_metrics[metric] = normalized
        confidence_intervals[f'{metric}_ci'] = conf_interval
    
    # Add normalized metrics and confidence intervals to DataFrame
    for metric in metrics:
        grouped_df[metric] = normalized_metrics[metric]
        grouped_df[f'{metric}_ci'] = confidence_intervals[f'{metric}_ci']
    
    # Calculate composite score
    grouped_df['composite_score'] = grouped_df[metrics].mean(axis=1)
    
    # Sort by composite score
    grouped_df = grouped_df.sort_values('composite_score', ascending=False)
    
    # Round specific columns
    round_cols = metrics + [m + '_ci' for m in metrics] + ['composite_score', 'minutes_played']
    grouped_df[round_cols] = grouped_df[round_cols].round(2)
    
    # Set goalkeeper as index
    return grouped_df.set_index('goalkeeper')

In [ ]:
def analyze_multiple_leagues(leagues):
    """
    Analyze goalkeeper performance across multiple leagues and seasons
    
    Parameters:
    leagues (list): List of dictionaries containing competition_id and season_ids
    
    Returns:
    pandas.DataFrame: Consolidated goalkeeper analysis across all leagues
    """
    all_results = []
    
    for league in leagues:
        competition_id = league['competition_id']
        season_ids = league['season_ids']
        
        try:
            competition_name = sb.competitions()[
                sb.competitions()['competition_id'] == competition_id
            ]['competition_name'].iloc[0]
        except:
            competition_name = f"Competition {competition_id}"
            
        logging.info(f"Processing {competition_name}")
        
        for season_id in tqdm(season_ids, desc=f"Processing seasons for {competition_name}"):
            try:
                logging.info(f"Analyzing season {season_id}")
                
                # Get season analysis
                season_analysis = analyze_goalkeeper_performance(competition_id, season_id)
                
                if not season_analysis.empty:
                    # Add competition and season information
                    season_analysis = season_analysis.reset_index()
                    season_analysis['competition_id'] = competition_id
                    season_analysis['competition_name'] = competition_name
                    season_analysis['season_id'] = season_id
                    
                    all_results.append(season_analysis)
                
            except Exception as e:
                logging.error(f"Error processing season {season_id}: {str(e)}")
                continue
    
    if not all_results:
        logging.error("No results found for any league/season")
        return pd.DataFrame()
    
    # Combine all results
    combined_df = pd.concat(all_results, ignore_index=True)
    
    # For each goalkeeper, keep only the entry with the most minutes played
    consolidated_df = (combined_df
        .sort_values('minutes_played', ascending=False)
        .groupby('goalkeeper')
        .first()
        .reset_index()
    )
    
    # Sort by composite score
    consolidated_df = consolidated_df.sort_values('composite_score', ascending=False)
    
    # Select and reorder columns
    columns = [
        'goalkeeper',
        'team',
        'competition_name',
        'season_id',
        'matches_played',
        'minutes_played',
        'shot_stopping',
        'distribution',
        'sweeping',
        'aerial_command',
        'composite_score'
    ]
    
    # Round numeric columns
    numeric_columns = [
        'minutes_played',
        'shot_stopping',
        'distribution',
        'sweeping',
        'aerial_command',
        'composite_score'
    ]
    
    final_df = consolidated_df[columns].copy()
    final_df[numeric_columns] = final_df[numeric_columns].round(2)
    
    return final_df

if __name__ == "__main__":
    # Your leagues list
    leagues = [
        {"competition_id": 9, "season_ids": [281]},
        {"competition_id": 43, "season_ids": [106]},
        {"competition_id": 11, "season_ids": [90, 42, 4]},
        {"competition_id": 7, "season_ids": [235, 108]},
        {"competition_id": 2, "season_ids": [44]},
        {"competition_id": 12, "season_ids": [27]},
        {"competition_id": 55, "season_ids": [282]},
    ]
    
    try:
        logging.info("Starting multi-league goalkeeper analysis")
        
        # Run the analysis
        results = analyze_multiple_leagues(leagues)
        
        if not results.empty:
            # Save to single CSV file
            output_filename = "goalkeeper_analysis_consolidated.csv"
            results.to_csv(output_filename, index=False)
            logging.info(f"Analysis completed. Results saved to {output_filename}")
            
            # Print summary statistics
            print("\nAnalysis Summary:")
            print(f"Total goalkeepers analyzed: {len(results)}")
            print(f"\nTop 5 goalkeepers by composite score:")
            print(results[['goalkeeper', 'team', 'competition_name', 'composite_score']].head())
            
        else:
            logging.error("No results generated")
            
    except Exception as e:
        logging.error(f"Error during analysis: {str(e)}")
        raise